# Comparing Classical Specular-Highlight Detection Algorithms

This notebook implements **four** classical (pre-deep-learning) single-image specular
detection/separation algorithms and compares them side by side on randomly sampled images
from any dataset folder, following the same `URL_PATH` + random-sample pattern as
`test_shafer_dichromatic_specular.ipynb`.

## Important caveats about the source table (read this first)

The user-supplied comparison table named 6 "generations" of dichromatic-model algorithms.
Before implementing, each was checked against the literature:

| # | Table's name | Real paper? | Included here? |
|---|---|---|---|
| 1 | Shafer / Klinker (1985-1990) | Yes (Shafer 1985 dichromatic model; Klinker et al.) | **Yes** — our validated `classical_specular_mask.py` |
| 2 | Tan & Ikeuchi (2005) | Yes, real PAMI 2005 paper | **Yes** — simplified single-pass version |
| 3 | Shen et al. ("Color-Lines") | Real: Shen/Zhang/Shao/Xin 2008 (chromaticity) + Shen & Zheng 2013 (intensity ratio) | **Yes** — simplified intensity-ratio + coarse chromaticity binning |
| 4 | Kim / Guo (Dark Channel / Low-Rank) | Real: Kim et al. CVPR 2013 (dark channel prior); Guo et al. ECCV 2018 (sparse+low-rank) | **Yes** — simplified dark-channel-elevation version |
| 5 | Hardware polarization / Stokes | Real research direction, but needs a **polarization camera** | **No** — our datasets are plain RGB; no polarization data exists to test this on |
| 6 | "3D Volume DRM" (Gaussian Splatting / SSR-GS) | Real direction, but is NOT a single-2D-image algorithm | **No** (as a per-image test) — needs a **trained 3D model + multiple views**, which is what our own `spec-fastgs/tools/extract_reflection_score.py` already does. See the last markdown cell. |

**On the table's PSNR/FPS numbers:** PSNR is a *reflection-removal* quality metric (compares
a highlight-removed image against a clean ground-truth diffuse image). We don't have
ground-truth diffuse images for arbitrary real photos, and our actual use case is
*detection* (producing a locator mask for training supervision), not removal. So this
notebook does **not** try to reproduce those specific numbers — it measures our own,
actually-run numbers on our own data: **measured wall-clock speed**, **flagged-pixel %**,
and **pairwise agreement between methods** (a real, ground-truth-free way to see which
methods correlate and which are outliers).

**Implementation honesty note:** algorithms 2-4 below are *simplified, single-pass*
implementations of each paper's *core idea* — not full reproductions of the published
pipelines (which include iterative refinement, segmentation, or optimization steps not
replicated here). Each algorithm cell states exactly what is simplified.

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import grey_opening, minimum_filter

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "spec-fastgs"))
try:
    from tools.classical_specular_mask import specular_score as shafer_score
    USING_CANONICAL_IMPL = True
except Exception as e:
    USING_CANONICAL_IMPL = False
    print(f"[info] could not import canonical tools/classical_specular_mask.py ({e}); "
          f"using an inline copy of the Shafer+top-hat logic instead.")

    def _disk_footprint(radius):
        y, x = np.ogrid[-radius:radius + 1, -radius:radius + 1]
        return (x ** 2 + y ** 2 <= radius ** 2)

    def shafer_score(img01, sat_thresh=0.25, val_thresh=0.75, tophat_radius=12, tophat_thresh=0.08):
        maxc = img01.max(axis=-1); minc = img01.min(axis=-1)
        V = maxc
        S = np.where(maxc > 1e-6, (maxc - minc) / (maxc + 1e-6), 0.0)
        raw_mask = (V > val_thresh) & (S < sat_thresh)
        opened = grey_opening(V, footprint=_disk_footprint(tophat_radius))
        tophat = np.clip(V - opened, 0.0, None)
        mask = raw_mask & (tophat > tophat_thresh)
        score = tophat * raw_mask
        if score.max() > 0:
            score = score / score.max()
        return mask, score

print(f"repo root       : {REPO_ROOT}")
print(f"canonical impl? : {USING_CANONICAL_IMPL}")

## Config — EDIT THIS CELL

In [ ]:
# ============================================================
# EDIT ME
# ============================================================
URL_PATH = r"dataset\mipnerf360\counter\images"
# URL_PATH = r"dataset\Anisotropic-Synthetic-Dataset\teapot\train"

NUM_SAMPLES = 10
RANDOM_SEED = None      # None = truly random each run; set an int for reproducibility
MAX_DISPLAY_SIZE = 500  # longest side (px) for computation + display; keeps this fast
# ============================================================

IMAGE_EXTS = ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG")

## 1. Sample images

In [ ]:
folder = Path(URL_PATH)
if not folder.is_absolute():
    folder = REPO_ROOT / folder
if not folder.is_dir():
    raise FileNotFoundError(f"URL_PATH does not exist: {folder}")

all_images = set()
for pat in IMAGE_EXTS:
    all_images.update(folder.glob(pat))
all_images = sorted(all_images)
if not all_images:
    raise FileNotFoundError(f"No images found directly inside: {folder}")

rng = random.Random(RANDOM_SEED)
n = min(NUM_SAMPLES, len(all_images))
sampled = rng.sample(all_images, n)

def load_resized(path, max_size):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    m = max(w, h)
    if m > max_size:
        s = max_size / m
        img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    return img

images01 = [np.array(load_resized(p, MAX_DISPLAY_SIZE)).astype(np.float32) / 255.0 for p in sampled]
names = [p.name for p in sampled]
print(f"folder   : {folder}")
print(f"total    : {len(all_images)}")
print(f"sampled  : {names}")

## 2. Algorithm implementations

Every function has the SAME signature: `(img01: [H,W,3] float in [0,1]) -> (mask: bool[H,W], score: float[H,W] in [0,1])`.

In [ ]:
# ---- Algorithm 1: Shafer (1985) / Klinker dichromatic model ----
# Our validated production implementation: bright (V) + desaturated (S) threshold, gated
# by a morphological white top-hat blob filter (rejects flat bright-diffuse backgrounds
# and object-silhouette edges — see spec-fastgs/tools/classical_specular_mask.py for the
# full validated derivation). This is a widely-used SIMPLIFIED form of the dichromatic
# principle (not Klinker's original 3D-RGB-cube "dog-leg" cluster-fitting procedure, which
# requires segmenting the image into homogeneous-material regions first).
def algo1_shafer(img01):
    t0 = time.perf_counter()
    mask, score = shafer_score(img01)
    return mask, score, time.perf_counter() - t0


# ---- Algorithm 2: Tan & Ikeuchi (2005) specular-free image ----
# Core idea: Tan & Ikeuchi's specular-free image sets every pixel's maximum chromaticity
# to a fixed constant; the standard, widely-cited SIMPLIFIED practical realization of this
# (confirmed against the literature) is to subtract the per-pixel MINIMUM channel:
#   I'_c(x) = I_c(x) - Imin(x)
# Under the dichromatic model with a white/near-neutral illuminant, the specular component
# adds ~equally to all 3 channels, so Imin(x) approximates the specular "pedestal" at that
# pixel — a saturated diffuse color (e.g. pure red) has Imin~0 by definition (no specular
# contribution needed to explain it), while a true highlight (all channels lifted toward
# white) has a large Imin. So Imin(x) is the raw score (I - I' = Imin exactly).
#
# HISTORY (debugged live, see the notebook's execution history / chat log):
#  (a) an earlier draft subtracted Imax instead of Imin, degenerating to plain brightness
#      thresholding (63% flagged);
#  (b) fixed to Imin per the literature, plus a "bright_floor" gate (a pixel must also be
#      bright in absolute terms, matching our Shafer detector's V+S combination) — but
#      STILL flags ~15% mean / 25% max on counter;
#  (c) visually confirmed root cause: unlike Algorithm 1, this per-pixel test has NO
#      spatial-compactness constraint, so it cannot distinguish a small TRUE highlight
#      from a LARGE FLAT white/bright diffuse surface (walls, plastic bottles, paper) —
#      both satisfy "bright + achromatic" identically at the pixel level. This is the
#      SAME confound our top-hat gate was built to fix in Algorithm 1 (see
#      results/MIPNERF_BOTTLENECK_PLAN_2026-07-01.md §6), now independently reproduced by
#      a DIFFERENT algorithm family. Left as-is (not further tuned) because this is a
#      genuine, informative structural finding, not a bug to hide via threshold search —
#      see Section 7's discussion.
def algo2_tan_ikeuchi(img01, thresh=0.35, bright_floor=0.6):
    t0 = time.perf_counter()
    Imin = img01.min(axis=-1)
    Imax = img01.max(axis=-1)
    score = Imin  # already in [0,1]
    mask = (score > thresh) & (Imax > bright_floor)
    return mask, score, time.perf_counter() - t0


# ---- Algorithm 3: Shen et al. chromaticity / intensity-ratio ("color-lines") ----
# Core idea: for a DIFFUSE pixel, the intensity ratio Imax/(Imax-Imin) depends only on
# surface chromaticity (constant for a given material) and is independent of shading/
# geometry; it rises toward its max as specular contamination increases (assumes a
# uniform/white illuminant — the table's stated vulnerability). Pixels are grouped into a
# coarse pseudo-chromaticity grid (a lightweight stand-in for the paper's clustering step);
# each cluster's ROBUST (median) ratio is its diffuse baseline; pixels whose ratio exceeds
# their cluster's baseline by a margin are flagged specular. SIMPLIFIED: coarse binning
# instead of the paper's proper unsupervised clustering.
def algo3_shen_colorlines(img01, bins=16, margin=0.12):
    t0 = time.perf_counter()
    R, G, B = img01[..., 0], img01[..., 1], img01[..., 2]
    s = R + G + B + 1e-6
    r, g = R / s, G / s
    Imax = img01.max(axis=-1); Imin = img01.min(axis=-1)
    ratio = Imax / (Imax - Imin + 1e-6)
    rb = np.clip((r * bins).astype(int), 0, bins - 1)
    gb = np.clip((g * bins).astype(int), 0, bins - 1)
    bin_id = (rb * bins + gb).reshape(-1)
    flat_ratio = ratio.reshape(-1)
    baseline = np.full(bins * bins, np.nan)
    for b in np.unique(bin_id):
        vals = flat_ratio[bin_id == b]
        if len(vals) > 5:
            baseline[b] = np.median(vals)
    baseline_map = baseline[bin_id].reshape(ratio.shape)
    deviation = np.nan_to_num(ratio - baseline_map, nan=0.0)
    deviation = np.clip(deviation, 0.0, None)
    score = deviation / deviation.max() if deviation.max() > 0 else deviation
    mask = score > margin
    return mask, score, time.perf_counter() - t0


# ---- Algorithm 4: Kim et al. (2013) / Guo et al. dark channel prior ----
# Core idea (He et al.'s Dark Channel Prior, adapted by Kim et al. CVPR 2013 to specular
# separation): in most natural-image LOCAL PATCHES, some pixel/channel is dark somewhere
# in the patch. An added, roughly-achromatic specular component LIFTS this local minimum
# ('dark channel') well above its usual near-zero value. This extends Algorithm 2 above
# (a PER-PIXEL min-channel estimate) by adding a local PATCH minimum for spatial
# robustness — the genuine methodological distinction between the two papers, and it DOES
# show up empirically: Algorithm 4's flagged% (mean ~5.5%) sits far below Algorithm 2's
# (~15%) on the identical images, evidence that even simple patch-pooling meaningfully
# helps versus a pure per-pixel test. SIMPLIFIED: we use the raw elevated dark channel as
# the specular score; the full papers add an iterative MAP-based recovery step (Kim et
# al.) or a sparse+low-rank decomposition (Guo et al.), not reproduced here.
def algo4_dark_channel(img01, patch_radius=7, elevation_thresh=0.35, bright_floor=0.6):
    t0 = time.perf_counter()
    per_pixel_min = img01.min(axis=-1)
    dark_channel = minimum_filter(per_pixel_min, size=2 * patch_radius + 1)
    Imax = img01.max(axis=-1)
    score = dark_channel  # already in [0,1]
    mask = (score > elevation_thresh) & (Imax > bright_floor)
    return mask, score, time.perf_counter() - t0


ALGORITHMS = {
    "1. Shafer/Klinker": algo1_shafer,
    "2. Tan-Ikeuchi": algo2_tan_ikeuchi,
    "3. Shen ColorLines": algo3_shen_colorlines,
    "4. Kim/Guo DarkChannel": algo4_dark_channel,
}
print("Algorithms ready:", list(ALGORITHMS.keys()))


## 3. Run all algorithms on all sampled images

In [ ]:
records = []
results = {name: {"masks": [], "scores": []} for name in ALGORITHMS}

for img_idx, (img01, name) in enumerate(zip(images01, names)):
    for algo_name, fn in ALGORITHMS.items():
        mask, score, elapsed = fn(img01)
        results[algo_name]["masks"].append(mask)
        results[algo_name]["scores"].append(score)
        records.append({
            "image": name,
            "algorithm": algo_name,
            "flagged_%": 100.0 * mask.mean(),
            "time_ms": elapsed * 1000.0,
        })

df = pd.DataFrame(records)
print(f"Ran {len(ALGORITHMS)} algorithms x {len(images01)} images = {len(df)} results.")

## 4. Visual comparison grid

Each row is one sampled image; columns are the original followed by each algorithm's
red-overlay mask. Look for: does the mask trace real highlights (glints, sheens) or does
it flood flat bright regions (background, matte white objects) — the known failure mode
for ANY appearance-only detector?

In [ ]:
n_imgs = len(images01)
n_algos = len(ALGORITHMS)
fig, axes = plt.subplots(n_imgs, n_algos + 1, figsize=(3.2 * (n_algos + 1), 3.2 * n_imgs))
if n_imgs == 1:
    axes = axes[None, :]

for i, (img01, name) in enumerate(zip(images01, names)):
    axes[i, 0].imshow(img01)
    axes[i, 0].set_title(f"{name}\noriginal", fontsize=8)
    axes[i, 0].axis("off")
    for j, algo_name in enumerate(ALGORITHMS):
        mask = results[algo_name]["masks"][i]
        overlay = (img01 * 255).astype(np.uint8).copy()
        overlay[mask] = [255, 0, 0]
        pct = 100.0 * mask.mean()
        axes[i, j + 1].imshow(overlay)
        axes[i, j + 1].set_title(f"{algo_name}\n{pct:.2f}% flagged", fontsize=8)
        axes[i, j + 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Quantitative comparison: flagged % and measured speed

In [ ]:
summary = df.groupby("algorithm").agg(
    mean_flagged_pct=("flagged_%", "mean"),
    std_flagged_pct=("flagged_%", "std"),
    min_flagged_pct=("flagged_%", "min"),
    max_flagged_pct=("flagged_%", "max"),
    mean_time_ms=("time_ms", "mean"),
).round(3)
display(summary)
print("\nA healthy true-highlight scene usually flags ~1-5% of pixels. A method whose "
      "max_flagged_pct jumps far above the others on the SAME images is either much more "
      "sensitive (tune its threshold) or is being fooled by bright-diffuse regions.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

df.boxplot(column="flagged_%", by="algorithm", ax=ax1, grid=False)
ax1.set_title("Flagged % distribution across sampled images")
ax1.set_ylabel("% pixels flagged")
ax1.tick_params(axis="x", rotation=30)

summary["mean_time_ms"].plot(kind="bar", ax=ax2, color="steelblue")
ax2.set_title("Measured mean runtime per image (this notebook, this machine)")
ax2.set_ylabel("ms/image")
ax2.tick_params(axis="x", rotation=30)

plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Pairwise agreement (IoU) — a ground-truth-free comparison

With no ground-truth specular mask, we can't compute precision/recall. But we CAN measure
how much the methods agree with each other: high mutual IoU across most pairs suggests a
real, robust signal; a method that's an outlier vs. all the others (either much lower IoU,
or a much larger flagged region overlapping everyone else's small ones) is either finding
something the others miss, or being fooled by something the others aren't.

In [ ]:
algo_names = list(ALGORITHMS.keys())
n_a = len(algo_names)
iou_matrix = np.zeros((n_a, n_a))

for a, b in combinations(range(n_a), 2):
    ious = []
    for i in range(n_imgs):
        ma = results[algo_names[a]]["masks"][i]
        mb = results[algo_names[b]]["masks"][i]
        inter = (ma & mb).sum()
        union = (ma | mb).sum()
        ious.append(inter / union if union > 0 else 0.0)
    iou_matrix[a, b] = iou_matrix[b, a] = np.mean(ious)
np.fill_diagonal(iou_matrix, 1.0)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(iou_matrix, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(n_a)); ax.set_xticklabels(algo_names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n_a)); ax.set_yticklabels(algo_names, fontsize=8)
for a in range(n_a):
    for b in range(n_a):
        ax.text(b, a, f"{iou_matrix[a,b]:.2f}", ha="center", va="center",
                color="white" if iou_matrix[a, b] < 0.5 else "black", fontsize=8)
plt.colorbar(im, label="mean IoU across sampled images")
plt.title("Pairwise mask agreement (IoU)")
plt.tight_layout()
plt.show()

## 7. How to read this comparison (no single number = "best")

Without ground-truth specular masks, there is no precision/recall/F1 to crown a winner.
Judge "best for our use case" (a training-time specular locator) using all of the above
together:

1. **Visual quality (Section 4)** — does the mask trace real glints/sheens, or flood flat
   bright regions (background, matte white objects)? This is the single most important
   check, since a locator that mislocates specularity actively hurts training (this is
   exactly what happened with our own earlier luminance-mask experiment, v2.5-R1).
2. **Flagged % (Section 5)** — healthy true-highlight scenes flag roughly 1-5% of pixels;
   a method that's wildly higher on the SAME images is either mis-tuned or fooled.
3. **Speed (Section 5)** — all four are meant to run as an OFFLINE preprocessing sweep
   (like `spec-fastgs/tools/gen_shafer_priors.py`), so absolute speed matters less than for
   a real-time application — but a 10-100x slower method is still a real cost across a
   240+ image scene sweep.
4. **Agreement (Section 6)** — a method that's an outlier (low IoU with everyone else)
   deserves closer visual inspection before trusting it as a locator.

### The key empirical finding from this comparison

On the counter (Mip-NeRF, real) scene, **spatial context matters more than the specific
per-pixel dichromatic formula.** Ranked by mean flagged %: Algorithm 3 (Shen ColorLines,
~0.1%, likely under-sensitive/low-recall given the coarse binning) < Algorithm 1 (Shafer +
top-hat, ~2.2%) < Algorithm 4 (Kim/Guo dark channel + local patch pooling, ~5.5%) <<
Algorithm 2 (Tan-Ikeuchi, pure per-pixel, ~15%, visually confirmed to flood flat white
walls/bottles/paper). The two methods that incorporate SOME spatial reasoning (top-hat
blob-shape gating for #1, local-patch pooling for #4) are far more robust to the
bright-diffuse confound than the two that are purely per-pixel (#2) or only coarsely
chromaticity-binned (#3). This independently validates, via a completely different
algorithm family, the exact same lesson our own project learned the hard way with the
v2.5-R1 luminance mask: **an appearance-only, spatially-blind test cannot distinguish a
compact true highlight from a large flat bright/achromatic surface, because both satisfy
identical per-pixel color statistics.** Any locator intended for real-scene training
supervision should incorporate spatial compactness (as our production choice does), not
just per-pixel color.

Our production choice (`spec-fastgs/tools/classical_specular_mask.py`, Algorithm 1 here)
was arrived at by exactly this kind of empirical validation — see
`results/MIPNERF_BOTTLENECK_PLAN_2026-07-01.md` §6-7 for the full derivation (naive
threshold → flat-background false positive → linear-high-pass fix → silhouette-edge
artifact → morphological top-hat fix, validated on a full 240+600 image sweep).

## Not implemented here: table items 5 and 6

**5. Hardware-assisted DRM (polarization / Stokes vectors).** This requires images captured
with a **polarization camera** (or a polarizing filter rotated across multiple exposures) to
measure the Stokes parameters, which directly disambiguate specular from diffuse light via
the physical polarization difference between them. Our datasets (Mip-NeRF 360, Anisotropic
Synthetic, NSVF) are all plain RGB photos/renders with **no polarization data** — there is
nothing for this algorithm to run on. It cannot be retrofitted from ordinary RGB images.

**6. "3D Volume DRM" (Gaussian Splatting / SSR-GS-style).** This is not a single-image
algorithm at all — it decouples view-dependent specular equations across a **trained 3D
Gaussian Splatting model observed from multiple calibrated camera views**, which is a
fundamentally different input (a converged 3D scene, not one 2D photo). This is exactly
what our own project already does: `spec-fastgs/utils/fast_utils.py`'s `spec_densify`
(model-residual locator) and `spec-fastgs/tools/extract_reflection_score.py` (a geometry-
gated, model-free multi-view Reflection Score) are both instances of this family. If you
want to compare against the four single-image methods above, the fair comparison is: train
a model on a scene, then run `extract_reflection_score.py` against it and inspect its
`<source>/ref_priors/*.png` outputs the same way as Section 4 above — that's a separate,
heavier (needs a trained model + GPU) experiment, not something to squeeze into this
single-image comparison notebook.